In [ ]:
pip install geopandas matplotlib shapely

In [ ]:
pip install plotly geopandas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import koreanize_matplotlib
import warnings

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우에서는 'Malgun Gothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS에서는 'AppleGothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'NanumGothic'  # Ubuntu와 같은 리눅스에서는 'NanumGothic'을 사용합니다.
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box

def create_grid(bounds, cell_size):
    """ 주어진 범위와 셀 크기로 격자 생성 """
    minx, miny, maxx, maxy = bounds
    x_coords = np.arange(minx, maxx, cell_size)
    y_coords = np.arange(miny, maxy, cell_size)
    
    grid_cells = []
    for x in x_coords:
        for y in y_coords:
            grid_cells.append(box(x, y, x + cell_size, y + cell_size))
    
    return grid_cells

# Shapefile 데이터 경로 설정
shapefile_path = r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\인천 데이터\인천시_인구통계_레이어_2024-06\6월\INCHEON_UMD_POPP_STAT.shp"

# Shapefile 데이터 로드
gdf = gpd.read_file(shapefile_path)

# SIDO_NM 고유값 확인
print("Unique SIDO_NM values:", gdf['SIDO_NM'].unique())

# 인천광역시 데이터 필터링
incheon_gdf = gdf[gdf['SIDO_NM'].str.contains('인천광', na=False)]
print(incheon_gdf.head())  # 데이터 확인

# 인천광역시의 경계 추출
bounds = incheon_gdf.total_bounds
print("Bounds:", bounds)  # 확인용

# 셀 크기 확인 (미터 단위가 맞는지 확인)
cell_size = 500  # 500미터

# 격자 생성
grid_cells = create_grid(bounds, cell_size)

# 격자 GeoDataFrame 생성
grid_gdf = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs=incheon_gdf.crs)

# 각 격자 셀의 중심점 추출
grid_gdf['centroid'] = grid_gdf['geometry'].centroid

# 인구수 데이터를 격자 셀에 집계
def aggregate_population(grid_gdf, population_gdf):
    """ 인구수 데이터를 격자 셀에 집계 """
    results = []
    for index, row in grid_gdf.iterrows():
        cell = row['geometry']
        # 'TOT_NMPR_C' 컬럼을 사용하여 인구수 집계
        population_in_cell = population_gdf[population_gdf['geometry'].within(cell)]['TOT_NMPR_C'].sum()
        results.append(population_in_cell)
    
    grid_gdf['population'] = results

# 인구수 집계
aggregate_population(grid_gdf, incheon_gdf)

# 플로팅
fig, ax = plt.subplots(figsize=(12, 12))

# 인천광역시 경계 시각화
incheon_gdf.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=1)

# 히트맵 시각화
grid_gdf.plot(column='population', ax=ax, edgecolor='black', cmap='OrRd', legend=True, alpha=0.6)

# 제목과 축 레이블 추가
plt.title('인천광역시 읍면동별 인구수 히트맵')
plt.xlabel('경도')
plt.ylabel('위도')

# 그래프 저장
plt.savefig('incheon_population_heatmap.png', dpi=300)
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Shapefile 데이터 경로 설정
shapefile_path = r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\인천 데이터\인천시_인구통계_레이어_2024-06\6월\INCHEON_UMD_POPP_STAT.shp"

# Shapefile 데이터 로드
gdf = gpd.read_file(shapefile_path)

# 데이터 확인
print(gdf.head())
print(gdf.columns)

# 인구 밀도를 나타내는 열을 사용 (TOT_NMPR_C 사용)
population_column = 'TOT_NMPR_C'

# 데이터 시각화 (히트맵)
fig, ax = plt.subplots(figsize=(12, 12))
gdf.plot(column=population_column, cmap='OrRd', linewidth=0.8, ax=ax, edgecolor='0.8')

# 컬러바 추가
sm = plt.cm.ScalarMappable(cmap='OrRd', norm=plt.Normalize(vmin=gdf[population_column].min(), vmax=gdf[population_column].max()))
sm._A = []
cbar = fig.colorbar(sm)

# 제목과 축 레이블 추가
plt.title('인천광역시 인구통계 히트맵')
plt.xlabel('경도')
plt.ylabel('위도')
plt.show()


In [ ]:
print(gdf['SIDO_NM'].unique())

In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import plotly.express as px
import plotly.graph_objects as go

# Shapefile 데이터 경로 설정
shapefile_path = r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\인천 데이터\인천시_인구통계_레이어_2024-06\6월\INCHEON_UMD_POPP_STAT.shp"

# Shapefile 데이터 로드
gdf = gpd.read_file(shapefile_path)

# 지도 영역 범위 구하기
xmin, ymin, xmax, ymax = gdf.total_bounds

# 500m x 500m 그리드 생성
grid_size = 500  # 500 meters
rows = np.arange(ymin, ymax + grid_size, grid_size)
cols = np.arange(xmin, xmax + grid_size, grid_size)
polygons = [box(x, y, x + grid_size, y + grid_size) for x in cols for y in rows]
grid = gpd.GeoDataFrame({'geometry': polygons}, crs=gdf.crs)

# 그리드 중심 좌표 계산
grid['x_center'] = grid.geometry.centroid.x
grid['y_center'] = grid.geometry.centroid.y

# Plotly를 사용하여 그리드를 지도에 표시
fig = go.Figure()

# 그리드 추가
for _, row in grid.iterrows():
    x0, y0, x1, y1 = row['geometry'].bounds
    fig.add_trace(go.Scattermapbox(
        mode = "lines",
        lon = [x0, x1, x1, x0, x0],
        lat = [y0, y0, y1, y1, y0],
        line = {'width': 1, 'color': 'blue'},
    ))

# 지도 설정
fig.update_layout(
    mapbox_style="open-street-map",
    mapbox=dict(
        center=go.layout.mapbox.Center(
            lat=(ymin + ymax) / 2,
            lon=(xmin + xmax) / 2,
        ),
        zoom=10,
    ),
    width=800,
    height=600,
    margin={"r":0,"t":0,"l":0,"b":0}
)

fig.show()
